# Multiple Linear Regression from Scratch: California Housing

Predicting `median_house_value` for California districts with **multiple linear regression implemented from scratch in NumPy** (closed-form Normal Equation). `scikit-learn` is used only for the train/test split and the R² metric.

**Result:** R² ≈ **0.64** on the held-out test set, in line with typical linear baselines on this dataset.

## Contents
1. Setup and data loading
2. Data cleaning and exploration
3. Train/test split and feature scaling
4. Model: Normal Equation
5. Evaluation
6. Conclusions and next steps

## Requirements
`numpy`, `pandas`, `matplotlib`, `scikit-learn`. Dataset: `housing.csv` (California Housing, 20,640 rows), placed in the same folder as this notebook.

## 1. Setup and data loading

In [61]:
import numpy as np                      # numerical computing / matrix algebra
import matplotlib.pyplot as plt         # plotting (not used yet, kept for future plots)
import pandas as pd                     # tabular data handling
from sklearn.metrics import r2_score    # evaluation metric

In [62]:
# Load the California Housing dataset (must be in the same folder as this notebook)
df = pd.read_csv("housing.csv")

In [63]:
# Preview the first 5 rows to understand the columns
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [64]:
# Dataset size: (rows, columns)
df.shape

(20640, 10)

## 2. Data cleaning and exploration

`df.info()` shows that `total_bedrooms` has **207 missing values** (20,433 non-null out of 20,640). Since that is only ~1% of the data, the affected rows are simply dropped.

In [65]:
# Column types and non-null counts, used to spot missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [66]:
# Drop rows with missing values (only total_bedrooms has any, ~1% of rows)
df = df.dropna()

In [67]:
# Confirm the row count after dropping: 20,640 -> 20,433
df.shape

(20433, 10)

In [68]:
# Check for duplicate rows
total_duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {total_duplicates}")

Total duplicate rows: 0


### Summary statistics

Things worth noticing in the output below:
- Features are on very different scales (e.g. `total_rooms` up to 39,320 vs `median_income` up to ~15), so **scaling is needed** for stable coefficients.
- `median_house_value` is **capped at 500,001**, which limits how well a linear model can fit the top end.

In [69]:
# Summary statistics (transposed so each feature is a row)
print(df.describe().T)

                      count           mean            std         min  \
longitude           20433.0    -119.570689       2.003578   -124.3500   
latitude            20433.0      35.633221       2.136348     32.5400   
housing_median_age  20433.0      28.633094      12.591805      1.0000   
total_rooms         20433.0    2636.504233    2185.269567      2.0000   
total_bedrooms      20433.0     537.870553     421.385070      1.0000   
population          20433.0    1424.946949    1133.208490      3.0000   
households          20433.0     499.433465     382.299226      1.0000   
median_income       20433.0       3.871162       1.899291      0.4999   
median_house_value  20433.0  206864.413155  115435.667099  14999.0000   

                            25%          50%         75%          max  
longitude             -121.8000    -118.4900    -118.010    -114.3100  
latitude                33.9300      34.2600      37.720      41.9500  
housing_median_age      18.0000      29.0000      37.

## 3. Train/test split and feature scaling

- **Features (X):** all 8 numeric columns. The categorical `ocean_proximity` column is dropped for simplicity (see next steps).
- **Target (y):** `median_house_value`.
- **Split:** 80% train / 20% test, with `random_state=42` for reproducibility.
- **Scaling:** standardization (z-score), with the mean and std computed on the **training set only** and then applied to the test set, to avoid data leakage.
- **Bias term:** a column of ones is added so the intercept is learned as `theta[0]`.

In [70]:
from sklearn.model_selection import train_test_split

# X: numeric features only (drop target and the categorical column)
X = df.drop(columns=["median_house_value", "ocean_proximity"]).values
# y: target variable
y = df['median_house_value']

# 80/20 train-test split; random_state makes the split reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,       # 20% test, 80% train
    random_state=42
)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Training features shape: (16346, 8)
Testing features shape: (4087, 8)


In [71]:
# Sanity check: number of training targets matches number of training rows
y_train.shape

(16346,)

In [72]:
# Peek at the raw (unscaled) test features
X_test

array([[-1.1724e+02,  3.2790e+01,  2.0000e+01, ...,  5.2500e+02,
         2.5400e+02,  3.1838e+00],
       [-1.2129e+02,  3.8010e+01,  2.0000e+00, ...,  3.3270e+03,
         9.5700e+02,  4.4871e+00],
       [-1.1814e+02,  3.3920e+01,  3.1000e+01, ...,  2.3130e+03,
         8.0100e+02,  3.2237e+00],
       ...,
       [-1.1903e+02,  3.5320e+01,  1.2000e+01, ...,  1.2940e+03,
         5.2300e+02,  2.5575e+00],
       [-1.1742e+02,  3.4080e+01,  2.8000e+01, ...,  1.3120e+03,
         4.4400e+02,  3.4844e+00],
       [-1.1710e+02,  3.2570e+01,  1.4000e+01, ...,  3.6620e+03,
         1.1930e+03,  2.3253e+00]], shape=(4087, 8))

In [73]:
def scale(A_tr, A_te):
    """Standardize features using TRAIN statistics only (prevents data leakage)."""
    m, s = A_tr.mean(0), A_tr.std(0)   # per-feature mean and std from the training set
    s[s == 0] = 1                      # avoid division by zero for constant columns
    return (A_tr - m) / s, (A_te - m) / s

# Prepend a column of ones so theta[0] acts as the intercept (bias)
add_bias = lambda A: np.hstack([np.ones((len(A), 1)), A])

X_train, X_test = scale(X_train, X_test)
Xtr, Xte = add_bias(X_train), add_bias(X_test)

In [74]:
# Shapes should now be (n_samples, 9): 8 features + 1 bias column
print(f"Training features shape: {Xtr.shape}")
print(f"Testing features shape: {Xte.shape}")

Training features shape: (16346, 9)
Testing features shape: (4087, 9)


## 4. Model: Normal Equation

Ordinary least squares has a closed-form solution, so no iterative optimization (gradient descent) is required:

$$\theta = (X^T X)^{-1} X^T y$$

where $X$ is the design matrix (with the bias column) and $y$ is the vector of house values.

In [75]:
# Initial parameter vector of zeros, shape (9, 1).
# (Only needed for gradient descent; it is overwritten by the Normal Equation below.)
theta_mat = np.zeros(Xtr.shape[1]).astype(float).reshape(-1, 1)

In [76]:
# Inspect the initial parameters
theta_mat

array([[0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.]])

In [77]:
# Solve for the parameters in closed form: theta = (X^T X)^-1 X^T y
theta_mat = np.linalg.inv(Xtr.T @ Xtr) @ Xtr.T @ y_train

In [78]:
# theta is a 1-D array of 9 values (1 intercept + 8 feature weights)
theta_mat.shape

(9,)

In [79]:
# Learned parameters as a column vector.
# Order: [intercept, longitude, latitude, housing_median_age, total_rooms,
#         total_bedrooms, population, households, median_income]
theta_mat.reshape(9, 1)

array([[206644.40009788],
       [-85341.725905  ],
       [-90434.67078093],
       [ 14527.41138834],
       [-18174.25470158],
       [ 48354.30740632],
       [-41513.2259601 ],
       [ 15739.49656088],
       [ 76467.23987025]])

### Interpreting the coefficients

Features were standardized, so each weight is the change in predicted house value (in $) for a **one standard deviation increase** in that feature, holding the others fixed.

- The intercept (~$206,644) is close to the mean house value, as expected after standardization.
- `median_income` has a large positive effect (~+$76k per std), the strongest driver of price.
- `longitude` and `latitude` have large negative weights, reflecting the location effect (coastal areas are more expensive).
- `total_rooms`/`total_bedrooms`/`households`/`population` are highly correlated with each other, so their individual signs (e.g. negative for `total_rooms` and `population`, but positive for `total_bedrooms`) should not be over-interpreted.

## 5. Evaluation

R² (coefficient of determination) measures the fraction of variance in house prices explained by the model on the unseen test set.

In [80]:
# Predict on the test set: y_hat = X @ theta
y_pred = Xte @ theta_mat

# R² score on the held-out test data
r2 = r2_score(y_test, y_pred)
print(f"R2 Score: {r2}")

R2 Score: 0.6400865688993737


## 6. Conclusions and next steps

The model explains roughly **64%** of the variance in house prices, a reasonable baseline for plain linear regression on this dataset.

The remaining error is mainly due to non-linear effects (especially location) and the price cap at $500,001.

**Ideas to improve:**
- One-hot encode `ocean_proximity` instead of dropping it
- Engineer features such as rooms per household, bedrooms per room, and population per household
- Log-transform skewed features (`total_rooms`, `population`, ...)
- Remove capped target values (`median_house_value == 500001`)
- Add polynomial/interaction terms, or try regularization (Ridge/Lasso)
- Compare with non-linear models (Random Forest, Gradient Boosting)
- Use `np.linalg.pinv` / `np.linalg.lstsq` instead of `inv` for better numerical stability